# Embedding：从查表、梯度到向量检索

## 学习目标

1. 理解 one-hot 乘矩阵为何等价于按 token id 查表。
2. 能写出输入 embedding 的形状与稀疏行梯度，并区分输出 softmax 的稠密梯度。
3. 掌握点积、余弦、L2、pooling、权重共享和向量检索的一致性约束。
4. 能定位 id 越界、PAD、零范数、dtype、维度和模型版本错配。

设词表大小为 $V$、向量维度为 $D$、输入形状为 $[B,L]$，embedding 表 $E\in\mathbb{R}^{V\times D}$，输出形状为 $[B,L,D]$。本 notebook 只依赖 NumPy。

In [ ]:
import math
import numpy as np

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(7)
vocab = ['<pad>', '猫', '狗', '汽车', '睡觉', '行驶']
token_to_id = {token: i for i, token in enumerate(vocab)}
V, D = len(vocab), 4
E = rng.normal(0.0, 0.2, size=(V, D)).astype(np.float32)
E[token_to_id['<pad>']] = 0.0
print('E.shape =', E.shape)
print('PAD row =', E[0])

## 1. one-hot 乘法与查表

若 $e_i$ 是第 $i$ 个 one-hot 行向量，则 $e_iE=E_{i,:}$。工程实现使用 gather，避免构造 $[B,L,V]$ 的巨大稀疏张量。下面同时计算两条路径并验证完全一致。

In [ ]:
token_ids = np.array([[1, 4, 0], [2, 3, 5]], dtype=np.int64)  # [B=2, L=3]
lookup = E[token_ids]
one_hot = np.eye(V, dtype=np.float32)[token_ids]
matmul = one_hot @ E
print('ids:', token_ids.shape, 'one_hot:', one_hot.shape, 'output:', lookup.shape)
print('max error =', np.max(np.abs(lookup - matmul)))
assert lookup.shape == (2, 3, D)
assert np.allclose(lookup, matmul)

## 2. 输入查表的反向传播

对上游梯度 $g_t$，表第 $i$ 行的梯度为

$$\frac{\partial\mathcal L}{\partial E_i}=\sum_{t:x_t=i}g_t.$$

同一 id 重复出现时必须累加；本 batch 未出现的行梯度为零。下面用 np.add.at 模拟 scatter-add。

In [ ]:
ids = np.array([1, 2, 1, 4])
upstream = np.arange(16, dtype=np.float32).reshape(4, D) / 10
grad_E = np.zeros_like(E)
np.add.at(grad_E, ids, upstream)
print('row 1 receives position 0 + position 2:', grad_E[1])
print('expected:', upstream[0] + upstream[2])
print('unused row 3:', grad_E[3])
assert np.allclose(grad_E[1], upstream[0] + upstream[2])

# 若 E 同时作为输出分类器，full softmax 会给所有行稠密梯度。
h = rng.normal(size=D)
logits = E @ h
prob = np.exp(logits - logits.max()); prob /= prob.sum()
target = 2
grad_logits = prob.copy(); grad_logits[target] -= 1
grad_output_E = grad_logits[:, None] * h[None, :]
print('output path nonzero rows =', np.count_nonzero(np.linalg.norm(grad_output_E, axis=1)))

## 3. 相似度、归一化与具体例子

点积同时受方向和范数影响；余弦只比较方向；单位向量上有 $\|x-y\|_2^2=2-2\cos(x,y)$，因此 cosine、inner product 与 L2 的排序等价。零向量归一化必须加 epsilon。

In [ ]:
def l2_normalize(x, eps=1e-12):
    norm = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.maximum(norm, eps)

q = np.array([1.0, 0.0])
candidates = np.array([[10.0, 1.0], [0.9, 0.0], [0.0, 2.0]])
dot = candidates @ q
cos = l2_normalize(candidates) @ l2_normalize(q)
l2 = np.linalg.norm(candidates - q, axis=1)
print('dot =', dot, 'ranking =', np.argsort(-dot))
print('cos =', cos, 'ranking =', np.argsort(-cos))
print('L2  =', l2, 'ranking =', np.argsort(l2))
print('说明：大范数会主导点积，但不会主导余弦。')

## 4. 从 token 状态得到句向量

Token embedding 是静态表的一行；上下文化隐藏状态依赖整句。句向量常用 masked mean 或模型指定的 CLS/last-token pooling。PAD 必须从分子和分母排除，且先平均再归一化与先逐 token 归一化再平均不同。

In [ ]:
def masked_mean(hidden, mask):
    weights = mask[..., None].astype(hidden.dtype)
    denom = weights.sum(axis=1)
    if np.any(denom == 0):
        raise ValueError('存在全 PAD 样本，不能做 mean pooling')
    return (hidden * weights).sum(axis=1) / denom

hidden = rng.normal(size=(2, 4, D)).astype(np.float32)
mask = np.array([[1, 1, 1, 0], [1, 1, 0, 0]], dtype=bool)
sent = l2_normalize(masked_mean(hidden, mask))
print('sentence embedding:', sent.shape)
print('norms:', np.linalg.norm(sent, axis=1))

## 5. 输入输出权重共享与大词表成本

共享时 logits 写成 $z=hE^\top+b$，省去另一张 $V\times D$ 输出矩阵，但 full softmax 的 $O(BLDV)$ 计算仍在。扩词必须同步 tokenizer、输入表、输出层和 vocab_size；只追加随机行却不给训练数据不会获得新语义。

In [ ]:
batch_hidden = rng.normal(size=(2, D)).astype(np.float32)
shared_logits = batch_hidden @ E.T
print('hidden', batch_hidden.shape, '@ E.T', E.T.shape, '-> logits', shared_logits.shape)
untied_parameters = 2 * V * D
tied_parameters = V * D
print('embedding/lm-head 参数（忽略 bias）:', untied_parameters, '->', tied_parameters)

## 6. 最小向量检索、量化与一致性

训练、建库、查询必须共享 encoder 版本、pooling、归一化、维度和 metric。下面用暴力 cosine 作为 exact 基线，再用 INT8 标量量化观察排序是否变化；真实 ANN 应把近似召回与模型语义召回分开测。

In [ ]:
docs = l2_normalize(rng.normal(size=(8, D)).astype(np.float32))
query = l2_normalize(rng.normal(size=(D,)).astype(np.float32))
exact_scores = docs @ query
exact_top3 = np.argsort(-exact_scores)[:3]

scale = max(np.max(np.abs(docs)), np.max(np.abs(query))) / 127.0
docs_q = np.clip(np.rint(docs / scale), -127, 127).astype(np.int8)
query_q = np.clip(np.rint(query / scale), -127, 127).astype(np.int8)
approx_scores = docs_q.astype(np.int32) @ query_q.astype(np.int32)
approx_top3 = np.argsort(-approx_scores)[:3]
print('exact top3 =', exact_top3, 'quantized top3 =', approx_top3)
print('top3 overlap =', len(set(exact_top3) & set(approx_top3)) / 3)

## 常见坑与工程变体

- tokenizer id 与权重行错位不会 shape 报错，却会彻底破坏模型；应保存协议哈希和 golden vectors。
- PAD 行为零不等于 PAD 已屏蔽；attention mask 和 label mask 仍必需。
- cosine 索引必须对文档和查询都归一化；零范数、NaN 和维度错误应在写索引前拒绝。
- HNSW 偏高召回/大内存，IVF 用聚类缩小扫描范围，PQ 用码本压缩；它们可组合。
- 句向量的 hard negative 可能是假负例；需要多正例、去重和人工抽检。

## 练习与面试总结

1. 修改 ids 让同一 token 出现三次，手算其 grad_E。
2. 比较 masked mean 与 last valid token 在左右 padding 下是否一致。
3. 将 D 扩大并测量暴力检索耗时、INT8 top-k 重合率。
4. 面试回答主线：$e_iE=E_i$ → $V\times D$ 参数与梯度 → 静态/上下文化 → metric 一致性 → 评估和版本化。